In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. Load the data
df = pd.read_csv('QB_Combine_Data.csv')

# 2. Parsing Function
def convert_to_inches(val):
    if pd.isna(val):
        return np.nan
    try:
        v = str(val).strip()
        if "'" in v:
            parts = v.replace('"', '').split("'")
            return int(parts[0]) * 12 + (int(parts[1]) if parts[1] else 0)
        return float(v)
    except:
        return np.nan

# Clean core physicals
df['Height_In'] = df['Height'].apply(convert_to_inches)
df['Broad_In'] = df['Broad'].apply(convert_to_inches)

# Identify the PFF column
pff_col = [col for col in df.columns if 'PFF' in col.upper()][0]
df[pff_col] = pd.to_numeric(df[pff_col], errors='coerce')

# 3. Explicit Feature List (Excluding Wingspan, Bench, and 10 Split)
features = [
    'Height_In', 'Weight', 'Hands', 'Arms', 
    '40 Yd', 'Vertical', 'Broad_In', '3-Cone', 'Shuttle'
]

# 4. Define x and y
y = df[pff_col]
x = df[features]

# 5. Preprocessing
# Fill missing drill results with the average (Imputation)
imputer = SimpleImputer(strategy='mean')
x_imputed = imputer.fit_transform(x)

# Normalize the data (Standardization) 
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_imputed)

# Rebuild the DataFrame for Statsmodels
x_final = pd.DataFrame(x_scaled, columns=features, index=df.index)
x_final = sm.add_constant(x_final)  # Add intercept

# 6. Run OLS Regression
model = sm.OLS(y, x_final).fit()
print(model.summary())


"""
Comments and some code written by Google Gemini Pro 3.1
"""

                              OLS Regression Results                              
Dep. Variable:      PFF_for_first_4_years   R-squared:                       0.289
Model:                                OLS   Adj. R-squared:                  0.052
Method:                     Least Squares   F-statistic:                     1.218
Date:                    Sun, 19 Apr 2026   Prob (F-statistic):              0.325
Time:                            17:45:37   Log-Likelihood:                -134.07
No. Observations:                      37   AIC:                             288.1
Df Residuals:                          27   BIC:                             304.2
Df Model:                               9                                         
Covariance Type:                nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       

'\nComments and some code written by Google Gemini Pro 3.1\n'

In [46]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as p

# 1. Load the data
df = pd.read_csv('QB_Combine_Data.csv')

# 2. Helper function to parse heights/broad jumps
def convert_to_inches(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    try:
        v = str(val).strip()
        if "'" in v:
            parts = v.replace('"', '').split("'")
            return int(parts[0]) * 12 + (int(parts[1]) if parts[1] else 0)
        return float(v)
    except:
        return np.nan

# Clean Height and Broad
col_height = [c for c in df.columns if 'Height' in c][0]
col_broad = [c for c in df.columns if 'Broad' in c][0]
df['Height_Inches'] = df[col_height].apply(convert_to_inches)
df['Broad_Inches'] = df[col_broad].apply(convert_to_inches)

# Force remaining combine attributes to numeric
pff_col = [col for col in df.columns if 'PFF' in col.upper()][0]
df[pff_col] = pd.to_numeric(df[pff_col], errors='coerce')

features = [
    'Weight', 'Hands', 'Arms', '40 Yd', 
    'Vertical', '3-Cone', 'Shuttle'
]

# Ensure we are using the cleaned column names for the model
actual_features = ['Height_Inches', 'Broad_Inches'] + [[c for c in df.columns if attr in c][0] for attr in features if [c for c in df.columns if attr in c]]

# Drop any rows where the PFF score itself is missing (we can't train on those)
df = df.dropna(subset=[pff_col])

# 3. Create the Binary Target for Logistic Regression
# 1 = Above Median PFF, 0 = Below Median PFF
median_pff = df[pff_col].median()
y = (df[pff_col] >= median_pff).astype(int)
x = df[actual_features]

# 4. Preprocessing: Impute missing values and scale the data
imputer = SimpleImputer(strategy='mean')
x_imputed = imputer.fit_transform(x)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_imputed)

# Create a clean dataframe for statsmodels
x_final = pd.DataFrame(x_scaled, columns=actual_features)

# Add a constant (intercept) to the model, which is required for statsmodels
x_final = sm.add_constant(X_final)

# 5. Build and train the Logit model

logit_model = sm.Logit(y, x_final)
result = logit_model.fit()
print(result.summary())

'\nComments and some code written by Google Gemini Pro 3.1\n'

Optimization terminated successfully.
         Current function value: 0.422507
         Iterations 7
                             Logit Regression Results                             
Dep. Variable:      PFF_for_first_4_years   No. Observations:                   37
Model:                              Logit   Df Residuals:                       27
Method:                               MLE   Df Model:                            9
Date:                    Sun, 19 Apr 2026   Pseudo R-squ.:                  0.3823
Time:                            17:50:15   Log-Likelihood:                -15.633
converged:                           True   LL-Null:                       -25.308
Covariance Type:                nonrobust   LLR p-value:                   0.02238
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.3002      0.491      0.612      0.541      -0.662 

'\nComments and some code written by Google Gemini Pro 3.1\n'

In [45]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

# 1. Load data
df = pd.read_csv('QB_Combine_Data.csv')

# 2. Convert height/broad
def convert_to_inches(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    try:
        v = str(val).strip().replace('"', '')
        if "'" in v:
            f, i = v.split("'")
            return int(f) * 12 + (int(i) if i else 0)
        return float(v)
    except:
        return np.nan

df['Height_Inches'] = df[[c for c in df.columns if 'Height' in c][0]].apply(convert_to_inches)
df['Broad_Inches'] = df[[c for c in df.columns if 'Broad' in c][0]].apply(convert_to_inches)

# 3. Target (median split)
pff_col = [c for c in df.columns if 'PFF' in c.upper()][0]
df[pff_col] = pd.to_numeric(df[pff_col], errors='coerce')
df = df.dropna(subset=[pff_col])

y = (df[pff_col] >= df[pff_col].median()).astype(int)

# 4. Features
features = ['Height_Inches', 'Broad_Inches', 'Weight', 'Hands', 'Arms',
            '40 Yd', 'Vertical', '3-Cone', 'Shuttle']

# 5. Impute + scale
imputer = SimpleImputer(strategy='mean')
x_imputed = imputer.fit_transform(x)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_imputed)

x_scaled_df = pd.DataFrame(x_scaled, columns=features)

# 6. LASSO FEATURE SELECTION
lasso = Lasso(alpha=0.1)
lasso.fit(x_scaled_df, y)

lasso_coefs = pd.Series(lasso.coef_, index=features)
selected_features = lasso_coefs[lasso_coefs != 0].index.tolist()

# 7. Build FINAL dataset using ONLY selected features
x_final = x_scaled_df[selected_features]
x_final = sm.add_constant(x_final)

# 8. Logit model
logit_model = sm.Logit(y, x_final)
result = logit_model.fit()

print(result.summary())

'\nComments and some code written by Google Gemini Pro 3.1\n'

Optimization terminated successfully.
         Current function value: 0.531210
         Iterations 6
                             Logit Regression Results                             
Dep. Variable:      PFF_for_first_4_years   No. Observations:                   37
Model:                              Logit   Df Residuals:                       34
Method:                               MLE   Df Model:                            2
Date:                    Sun, 19 Apr 2026   Pseudo R-squ.:                  0.2234
Time:                            17:50:07   Log-Likelihood:                -19.655
converged:                           True   LL-Null:                       -25.308
Covariance Type:                nonrobust   LLR p-value:                  0.003508
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.3150      0.388      0.812      0.417      -0.446    

'\nComments and some code written by Google Gemini Pro 3.1\n'